# A_1 — Duration and absorbing Markov chain

**Input:** `Results/08.csv`
**Output:** phase durations, success probabilities, bootstrap confidence intervals

Estimates the expected time from Consideration to commissioning. The lifecycle
is modelled as an absorbing Markov chain with phases 1–4 as transient states
and commissioned and cancelled as absorbing states; the expected duration is
the conditional mean first passage time to commissioning, which weights the time
spent in stalled phases and excludes the survivorship bias of averaging over
completed projects only. Confidence intervals come from 1,000 bootstrap
resamples.

Two funnel cohorts, tracked from phase 1 and from phase 2, are reported as a
robustness check against left-censoring.

In [11]:
import pandas as pd 
import numpy as np
import seaborn as sns
import ptitprince as pt

In [12]:
file_path = "Results/08.csv" 
df = pd.read_csv(file_path)

In [13]:
# Calculate the time spent in each state (Duration)
df['Duration'] = df['Tstop'] - df['Tstart']

# Create a readable string for transitions (e.g., "1 -> 2")
df['Transition'] = df['from'].astype(str) + " -> " + df['to'].astype(str)

# Sort the data logically by starting and ending state
df = df.sort_values(by=['from', 'to'])

In [14]:
# --- GLOBAL TRANSITION ANALYSIS (ALL PATHS) ---
all_transitions_stats = df.groupby('Transition')['Duration'].agg(['mean', 'count'])

all_transitions_stats = all_transitions_stats.sort_index()

all_transitions_stats.columns = ['Mean Duration (Years)', 'Sample Size (N)']

print("--- FULL TRANSITION STATISTICS ---")
print(all_transitions_stats)

print("\n--- SUMMARY BY OUTCOME TYPE ---")
success_mask = all_transitions_stats.index.isin(['1 -> 2', '2 -> 3', '3 -> 4', '4 -> 5'])
cancel_mask = all_transitions_stats.index.str.endswith('-> 6')

print(f"Total Success Path Transitions: {all_transitions_stats[success_mask]['Sample Size (N)'].sum()}")
print(f"Total Cancellation Transitions: {all_transitions_stats[cancel_mask]['Sample Size (N)'].sum()}")
print(f"Other Transitions (Backwards/Jumps): {all_transitions_stats[~success_mask & ~cancel_mask]['Sample Size (N)'].sum()}")

--- FULL TRANSITION STATISTICS ---
            Mean Duration (Years)  Sample Size (N)
Transition                                        
1 -> 1                   3.223881               67
1 -> 2                   2.600000              117
1 -> 3                   2.769231               13
1 -> 4                   2.666667                3
1 -> 5                   3.000000                2
1 -> 6                   3.285106              235
2 -> 1                   2.445161               31
2 -> 2                   3.563218               87
2 -> 3                   3.196262              107
2 -> 4                   3.058824               17
2 -> 5                   3.600000                5
2 -> 6                   3.186441              118
3 -> 1                   3.866667               15
3 -> 2                   2.727273               11
3 -> 3                   5.666667               54
3 -> 4                   4.169014              142
3 -> 5                   5.615385              

In [15]:
# --- SUCCESS PATH ANALYSIS (1 -> 5) ---
# Define the logical sequence of successful stages
target_success = ['1 -> 2', '2 -> 3', '3 -> 4', '4 -> 5']

# 1. Calculate the mean duration for each specific transition
success_means = df.groupby('Transition')['Duration'].mean().reindex(target_success)

# 2. Calculate cumulative duration (running sum)
cumulative_success = success_means.cumsum()

# 3. Count observations (Sample Size) per phase
success_counts = df.groupby('Transition').size().reindex(target_success).fillna(0).astype(int)

print("--- RAW MEANS PER PHASE (SUCCESS) ---")
print(success_means)

print("\n--- CUMULATIVE DURATION (TOTAL TIMELINE) ---")
print(cumulative_success)

print("\n--- SAMPLE SIZE PER PHASE ---")
print(success_counts)
print(f"\nTotal transitions in success path: {success_counts.sum()}")

--- RAW MEANS PER PHASE (SUCCESS) ---
Transition
1 -> 2    2.600000
2 -> 3    3.196262
3 -> 4    4.169014
4 -> 5    3.089041
Name: Duration, dtype: float64

--- CUMULATIVE DURATION (TOTAL TIMELINE) ---
Transition
1 -> 2     2.600000
2 -> 3     5.796262
3 -> 4     9.965276
4 -> 5    13.054317
Name: Duration, dtype: float64

--- SAMPLE SIZE PER PHASE ---
Transition
1 -> 2    117
2 -> 3    107
3 -> 4    142
4 -> 5    146
dtype: int64

Total transitions in success path: 512


### Mean First Passage Time (MFPT) - Markov chain

In [16]:
def compute_mfpt_for_sample(sample_df):
    # 1. Extract 'From_State'
    sample_df['From_State'] = sample_df['Transition'].astype(str).str.split(' -> ').str[0]
    sample_df['From_State'] = pd.to_numeric(sample_df['From_State'], errors='coerce')
    
    # 2. Calculate Total Exposure Time
    total_exposure = sample_df.groupby('From_State')['Duration'].sum()
    
    transient_states = [1, 2, 3, 4]
    absorbing_states = [5, 6]
    n_transient = len(transient_states)
    
    # 3. Build Count Matrix
    count_matrix = pd.DataFrame(0.0, index=transient_states, columns=transient_states + absorbing_states)
    stats_df = sample_df.groupby('Transition')['Duration'].agg(['count']).reset_index()
    stats_df[['From', 'To']] = stats_df['Transition'].astype(str).str.split(' -> ', expand=True)
    
    for _, row in stats_df.iterrows():
        if pd.notna(row['To']) and row['To'].strip().isdigit():
            i, j = int(row['From']), int(row['To'])
            if i in transient_states and j in (transient_states + absorbing_states):
                count_matrix.at[i, j] = row['count']
                
    # 4. Calculate Transition Probabilities
    total_transitions_out = count_matrix.sum(axis=1)
    
    # Avoid division by zero in empty bootstrap samples
    total_transitions_out = total_transitions_out.replace(0, 1) 
    P = count_matrix.div(total_transitions_out, axis=0)
    
    # 5. Calculate Waiting Times
    W = np.zeros(n_transient)
    for idx, state in enumerate(transient_states):
        exposure = total_exposure.get(state, 0)
        transitions = total_transitions_out.loc[state]
        if transitions > 0:
            W[idx] = exposure / transitions
            
    # 6 to 11. Core Markov Chain Operations
    Q = P.loc[transient_states, transient_states].values
    R = P.loc[transient_states, absorbing_states].values
    
    I = np.eye(n_transient)
    
    # Handle singular matrix errors that might occur during random resampling
    try:
        F = np.linalg.inv(I - Q)
    except np.linalg.LinAlgError:
        return np.full(n_transient, np.nan)
        
    B = np.dot(F, R)
    prob_reach_5 = B[:, 0]
    
    Q_star = np.zeros_like(Q)
    for i in range(n_transient):
        for j in range(n_transient):
            if prob_reach_5[i] > 0:
                Q_star[i, j] = Q[i, j] * (prob_reach_5[j] / prob_reach_5[i])
                
    try:            
        F_star = np.linalg.inv(I - Q_star)
    except np.linalg.LinAlgError:
        return np.full(n_transient, np.nan)
        
    mfpt_to_5 = np.dot(F_star, W)
    return mfpt_to_5

# Bootstrapping setup
n_iterations = 1000
bootstrap_mfpt_results = []

for iteration in range(n_iterations):
    # Resample the original dataframe with replacement
    df_resampled = df.sample(n=len(df), replace=True)
    
    # Compute MFPT for the resampled dataframe
    current_mfpt = compute_mfpt_for_sample(df_resampled)
    bootstrap_mfpt_results.append(current_mfpt)

# Convert results to a numpy array for percentile calculation
results_array = np.array(bootstrap_mfpt_results)

# Calculate the 95% Confidence Interval using nanpercentile to ignore failed inversions
lower_bound = np.nanpercentile(results_array, 2.5, axis=0)
upper_bound = np.nanpercentile(results_array, 97.5, axis=0)
mean_mfpt = np.nanmean(results_array, axis=0)

# Display results
transient_states = [1, 2, 3, 4]
print("\\n--- 95% CONFIDENCE INTERVALS FOR MFPT ---")
for idx, state in enumerate(transient_states):
    print(f"State {state} MFPT: {mean_mfpt[idx]:.1f} years (95% CI: [{lower_bound[idx]:.1f}, {upper_bound[idx]:.1f}])")

\n--- 95% CONFIDENCE INTERVALS FOR MFPT ---
State 1 MFPT: 14.1 years (95% CI: [13.0, 15.2])
State 2 MFPT: 11.9 years (95% CI: [11.0, 12.8])
State 3 MFPT: 8.8 years (95% CI: [8.2, 9.5])
State 4 MFPT: 3.8 years (95% CI: [3.4, 4.2])


In [17]:
def compute_mfpt_and_w_for_sample(sample_df):
    sample_df['From_State'] = sample_df['Transition'].astype(str).str.split(' -> ').str[0]
    sample_df['From_State'] = pd.to_numeric(sample_df['From_State'], errors='coerce')
    
    total_exposure = sample_df.groupby('From_State')['Duration'].sum()
    
    transient_states = [1, 2, 3, 4]
    absorbing_states = [5, 6]
    n_transient = len(transient_states)
    
    count_matrix = pd.DataFrame(0.0, index=transient_states, columns=transient_states + absorbing_states)
    stats_df = sample_df.groupby('Transition')['Duration'].agg(['count']).reset_index()
    stats_df[['From', 'To']] = stats_df['Transition'].astype(str).str.split(' -> ', expand=True)
    
    for _, row in stats_df.iterrows():
        if pd.notna(row['To']) and row['To'].strip().isdigit():
            i, j = int(row['From']), int(row['To'])
            if i in transient_states and j in (transient_states + absorbing_states):
                count_matrix.at[i, j] = row['count']
                
    total_transitions_out = count_matrix.sum(axis=1)
    
    total_transitions_out = total_transitions_out.replace(0, 1) 
    P = count_matrix.div(total_transitions_out, axis=0)
    
    W = np.zeros(n_transient)
    for idx, state in enumerate(transient_states):
        exposure = total_exposure.get(state, 0)
        transitions = total_transitions_out.loc[state]
        if transitions > 0:
            W[idx] = exposure / transitions
            
    Q = P.loc[transient_states, transient_states].values
    R = P.loc[transient_states, absorbing_states].values
    
    I = np.eye(n_transient)
    
    try:
        F = np.linalg.inv(I - Q)
    except np.linalg.LinAlgError:
        return np.full(n_transient, np.nan), np.full(n_transient, np.nan)
        
    B = np.dot(F, R)
    prob_reach_5 = B[:, 0]
    
    Q_star = np.zeros_like(Q)
    for i in range(n_transient):
        for j in range(n_transient):
            if prob_reach_5[i] > 0:
                Q_star[i, j] = Q[i, j] * (prob_reach_5[j] / prob_reach_5[i])
                
    try:            
        F_star = np.linalg.inv(I - Q_star)
    except np.linalg.LinAlgError:
        return np.full(n_transient, np.nan), np.full(n_transient, np.nan)
        
    mfpt_to_5 = np.dot(F_star, W)
    
    return mfpt_to_5, W

# --- Bootstrapping setup ---
n_iterations = 1000
bootstrap_mfpt_results = []
bootstrap_diff_results = [] 

for iteration in range(n_iterations):
    df_resampled = df.sample(n=len(df), replace=True)
    
    current_mfpt = compute_mfpt_for_sample(df_resampled)
    
    if np.isnan(current_mfpt).any():
        continue
        
    bootstrap_mfpt_results.append(current_mfpt)
    
    diff_1 = current_mfpt[0] - current_mfpt[1] 
    diff_2 = current_mfpt[1] - current_mfpt[2] 
    diff_3 = current_mfpt[2] - current_mfpt[3] 
    diff_4 = current_mfpt[3]                   
    
    bootstrap_diff_results.append([diff_1, diff_2, diff_3, diff_4])

mfpt_array = np.array(bootstrap_mfpt_results)
diff_array = np.array(bootstrap_diff_results)

diff_lower = np.nanpercentile(diff_array, 2.5, axis=0)
diff_upper = np.nanpercentile(diff_array, 97.5, axis=0)
diff_mean = np.nanmean(diff_array, axis=0)

phase_names = [
    "Consideration (MFPT 1 - MFPT 2)", 
    "Planned (MFPT 2 - MFPT 3)", 
    "Design & Permitting (MFPT 3 - MFPT 4)", 
    "Under Construction (MFPT 4)"
]

print("\n--- 95% CONFIDENCE INTERVALS ---")
for idx in range(4):
    print(f"{phase_names[idx]}: {diff_mean[idx]:.1f} years (95% CI: [{diff_lower[idx]:.1f}, {diff_upper[idx]:.1f}])")


--- 95% CONFIDENCE INTERVALS ---
Consideration (MFPT 1 - MFPT 2): 2.2 years (95% CI: [1.4, 3.0])
Planned (MFPT 2 - MFPT 3): 3.1 years (95% CI: [2.4, 3.8])
Design & Permitting (MFPT 3 - MFPT 4): 5.0 years (95% CI: [4.5, 5.7])
Under Construction (MFPT 4): 3.8 years (95% CI: [3.4, 4.2])


In [18]:
def compute_failure_probabilities(sample_df):
    sample_df['From_State'] = sample_df['Transition'].astype(str).str.split(' -> ').str[0]
    sample_df['From_State'] = pd.to_numeric(sample_df['From_State'], errors='coerce')
    
    transient_states = [1, 2, 3, 4]
    absorbing_states = [5, 6] 
    n_transient = len(transient_states)
    
    count_matrix = pd.DataFrame(0.0, index=transient_states, columns=transient_states + absorbing_states)
    stats_df = sample_df.groupby('Transition')['Duration'].agg(['count']).reset_index()
    stats_df[['From', 'To']] = stats_df['Transition'].astype(str).str.split(' -> ', expand=True)
    
    for _, row in stats_df.iterrows():
        if pd.notna(row['To']) and row['To'].strip().isdigit():
            i, j = int(row['From']), int(row['To'])
            if i in transient_states and j in (transient_states + absorbing_states):
                count_matrix.at[i, j] = row['count']
                
    total_transitions_out = count_matrix.sum(axis=1)
    total_transitions_out = total_transitions_out.replace(0, 1) # Prevent division by zero
    P = count_matrix.div(total_transitions_out, axis=0)
    
    Q = P.loc[transient_states, transient_states].values
    R = P.loc[transient_states, absorbing_states].values
    
    I = np.eye(n_transient)
    
    try:
        F = np.linalg.inv(I - Q)
    except np.linalg.LinAlgError:
        return np.full(n_transient, np.nan) 
        
    B = np.dot(F, R)
    
    failure_probabilities = B[:, 1] 
    
    return failure_probabilities

n_iterations = 1000
bootstrap_failure_results = []

for iteration in range(n_iterations):
    df_resampled = df.sample(n=len(df), replace=True)
    
    current_probs = compute_failure_probabilities(df_resampled)
    bootstrap_failure_results.append(current_probs)

failure_results_array = np.array(bootstrap_failure_results)

lower_bound = np.nanpercentile(failure_results_array, 2.5, axis=0)
upper_bound = np.nanpercentile(failure_results_array, 97.5, axis=0)
mean_failure_prob = np.nanmean(failure_results_array, axis=0)

transient_states = [1, 2, 3, 4]
print("\n--- 95% CONFIDENCE INTERVALS FOR FAILURE PROBABILITY ---")
for idx, state in enumerate(transient_states):
    mean_pct = mean_failure_prob[idx] * 100
    lower_pct = lower_bound[idx] * 100
    upper_pct = upper_bound[idx] * 100
    print(f"State {state} -> Failure: {mean_pct:.1f}% (95% CI: [{lower_pct:.1f}%, {upper_pct:.1f}%])")


--- 95% CONFIDENCE INTERVALS FOR FAILURE PROBABILITY ---
State 1 -> Failure: 87.0% (95% CI: [84.3%, 89.3%])
State 2 -> Failure: 69.3% (95% CI: [64.3%, 73.8%])
State 3 -> Failure: 42.1% (95% CI: [36.1%, 47.8%])
State 4 -> Failure: 14.1% (95% CI: [9.3%, 19.8%])


### Robustness Check: Left-Censoring Elimination via Funnel Cohort
Investments that we follwed from phase 1 to commissioning

In [19]:
PROJECT_COL = 'Inv_index' 

# --- 1. FUNNEL COHORT: ELIMINATING LEFT-CENSORING ---
# Find all unique projects that have a recorded '1 -> 2' transition.
# This guarantees we only track projects we have seen from the very beginning.
projects_starting_at_1 = df[df['Transition'] == '1 -> 2'][PROJECT_COL].unique()

df_funnel = df[df[PROJECT_COL].isin(projects_starting_at_1)].copy()

# --- 2. SUCCESS PATH ANALYSIS ON THE FUNNEL COHORT ---
target_success = ['1 -> 2', '2 -> 3', '3 -> 4', '4 -> 5']

funnel_means = df_funnel.groupby('Transition')['Duration'].mean().reindex(target_success)
funnel_cumulative = funnel_means.cumsum()
funnel_counts = df_funnel.groupby('Transition').size().reindex(target_success).fillna(0).astype(int)

# --- 3. PRINT RESULTS ---
print(f"Total projects in the dataset: {df[PROJECT_COL].nunique()}")
print(f"Projects starting from phase 1 (Funnel Base): {len(projects_starting_at_1)}\n")

print("--- FUNNEL COHORT: SAMPLE SIZE (SURVIVORS) PER PHASE ---")
print(funnel_counts)

print("\n--- FUNNEL COHORT: RAW MEANS PER PHASE ---")
print(funnel_means)

print("\n--- FUNNEL COHORT: CUMULATIVE DURATION ---")
print(funnel_cumulative)

Total projects in the dataset: 900
Projects starting from phase 1 (Funnel Base): 114

--- FUNNEL COHORT: SAMPLE SIZE (SURVIVORS) PER PHASE ---
Transition
1 -> 2    117
2 -> 3     32
3 -> 4     12
4 -> 5      7
dtype: int64

--- FUNNEL COHORT: RAW MEANS PER PHASE ---
Transition
1 -> 2    2.600000
2 -> 3    2.562500
3 -> 4    4.333333
4 -> 5    2.285714
Name: Duration, dtype: float64

--- FUNNEL COHORT: CUMULATIVE DURATION ---
Transition
1 -> 2     2.600000
2 -> 3     5.162500
3 -> 4     9.495833
4 -> 5    11.781548
Name: Duration, dtype: float64


### Robustness Check: Left-Censoring Elimination via Funnel Cohort
Investments that entered the report after in the second phase

In [20]:
# --- 0. CONFIGURATION ---
PROJECT_COL = 'Inv_index' 

projects_starting_at_2 = df[df['Transition'] == '2 -> 3'][PROJECT_COL].unique()
df_funnel_2 = df[df[PROJECT_COL].isin(projects_starting_at_2)].copy()

# --- 2. SUCCESS PATH ANALYSIS ON THE NEW FUNNEL ---
# We track only the path from phase 2 onwards
target_success = ['2 -> 3', '3 -> 4', '4 -> 5']

funnel_means = df_funnel_2.groupby('Transition')['Duration'].mean().reindex(target_success)
funnel_cumulative = funnel_means.cumsum()
funnel_counts = df_funnel_2.groupby('Transition').size().reindex(target_success).fillna(0).astype(int)

# --- 3. PRINT RESULTS ---
print(f"Total projects in the dataset: {df[PROJECT_COL].nunique()}")
print(f"Projects starting from phase 2 (Funnel Base): {len(projects_starting_at_2)}\n")

print("--- FUNNEL COHORT (FROM PHASE 2): SAMPLE SIZE PER PHASE ---")
print(funnel_counts)

print("\n--- FUNNEL COHORT (FROM PHASE 2): RAW MEANS PER PHASE ---")
print(funnel_means)

print("\n--- FUNNEL COHORT (FROM PHASE 2): CUMULATIVE DURATION ---")
print(funnel_cumulative)

Total projects in the dataset: 900
Projects starting from phase 2 (Funnel Base): 107

--- FUNNEL COHORT (FROM PHASE 2): SAMPLE SIZE PER PHASE ---
Transition
2 -> 3    107
3 -> 4     41
4 -> 5     23
dtype: int64

--- FUNNEL COHORT (FROM PHASE 2): RAW MEANS PER PHASE ---
Transition
2 -> 3    3.196262
3 -> 4    3.878049
4 -> 5    3.043478
Name: Duration, dtype: float64

--- FUNNEL COHORT (FROM PHASE 2): CUMULATIVE DURATION ---
Transition
2 -> 3     3.196262
3 -> 4     7.074310
4 -> 5    10.117789
Name: Duration, dtype: float64
